# HAR-12 with an isolated Suomi NPP reference

First run notebook 01 with `SENSORS = {'modis', 'viirs-snpp'}` for the required years. Its existing ZIP discovery supports `raw/VIIRS SNPP/viirs-snpp_<year>_all_countries.zip`. This notebook reads only the resulting processed files.

It constructs independent MODIS/S-NPP monthly tables and a separate S-NPP calibration. It does not consume or replace the NOAA-20 L1/L6 tables. Keep S-NPP out of the existing notebook 02 aggregation until that shared pipeline has an explicit satellite-selection policy.

The configured splice uses MODIS before January 2012 and S-NPP from January 2012. Defaults fit five calendar years on each side, with month effects, linear trends and a level change. Calibration uses only comparable MODIS/S-NPP months in 2018–2021. The pre-transition MODIS segment is scaled to S-NPP units. Missing years remain gaps.

Reported standard errors are conventional OLS model standard errors. They do not account for residual serial correlation. Results describe the configured splice and do not establish a causal sensor effect. Validate model diagnostics and real data before interpreting estimates.

The screenshot showed no 2016 or 2019 archive. Discovery reports missing processed years without replacing them with zero activity.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'], env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import re
import subprocess
import pandas as pd
from metrics import harmonization_snpp, harmonization_transition
from metrics.harmonization_factor import har06_factor

importlib.reload(harmonization_snpp)
importlib.reload(harmonization_transition)
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
GRID_DEG = 0.1
CAL_START, CAL_END, TEST_START = 2018, 2021, 2022
MIN_CAL_MONTHS = 24
MODEL_OPTIONS = dict(reference_sensor='viirs-snpp', transition_year=2012, window_years=5,
                     min_months_each_side=36, min_years_each_side=3, max_unknown_days=0)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
COLUMNS = ['country', 'instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.parquet.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

def atomic_json(value, path):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


In [ ]:
transition_year = MODEL_OPTIONS['transition_year']
calibration_years = set(range(CAL_START, CAL_END + 1))
required_years = {
    'modis': set(range(transition_year - MODEL_OPTIONS['window_years'], transition_year)) | calibration_years,
    'viirs-snpp': set(range(transition_year, transition_year + MODEL_OPTIONS['window_years'])) | calibration_years,
}
sources = []
for path in sorted(PROCESSED.glob('*.parquet')):
    match = re.fullmatch(r'(modis|viirs-snpp)_([0-9]{4})', path.stem)
    if match and int(match[2]) in required_years[match[1]]:
        sources.append((match[1], int(match[2]), path))
for sensor in ['modis', 'viirs-snpp']:
    if not any(source[0] == sensor for source in sources):
        raise FileNotFoundError(f'Missing processed {sensor} files in {PROCESSED}. Run notebook 01 first.')
missing_years = {}
for sensor in ['modis', 'viirs-snpp']:
    present = {year for selected_sensor, year, _ in sources if selected_sensor == sensor}
    missing_years[sensor] = sorted(required_years[sensor] - present)
print('Missing processed years, retained as gaps:', missing_years)
parts = []
for sensor, year, path in sources:
    frame = pd.read_parquet(path, columns=COLUMNS)
    dates = pd.to_datetime(frame.acq_date, errors='coerce', utc=True).dropna()
    if not dates.dt.year.eq(year).all():
        raise ValueError(f'Acquisition dates outside filename year in {path}')
    parts.append(harmonization_snpp.build_sensor_monthly(frame, sensor=sensor, grid_deg=GRID_DEG))
    del frame
source_monthly = pd.concat(parts, ignore_index=True)
monthly = source_monthly.drop(columns='sensor')
factors = har06_factor(monthly, min_cal_months=MIN_CAL_MONTHS, cal_start=CAL_START,
                       cal_end=CAL_END, test_start=TEST_START, max_unknown_days=MODEL_OPTIONS['max_unknown_days'])
result = harmonization_transition.har12_residual_step(monthly, factors, **MODEL_OPTIONS)


In [ ]:
METRICS.mkdir(parents=True, exist_ok=True)
parameters_path = METRICS / '_params_har12_snpp.json'
parameters_path.unlink(missing_ok=True)  # Invalidate an interrupted input bundle.
atomic_parquet(monthly, METRICS / 'l7_har12_snpp_monthly.parquet')
atomic_parquet(factors, METRICS / 'l7_har12_snpp_factor.parquet')
atomic_parquet(result, METRICS / 'l7_har12_residual_step.parquet')
parameters = dict(model_options=MODEL_OPTIONS, grid_deg=GRID_DEG, cal_start=CAL_START,
                  cal_end=CAL_END, min_cal_months=MIN_CAL_MONTHS, commit=COMMIT, missing_years=missing_years,
                  sources=[dict(sensor=sensor, year=year, name=path.name, size=path.stat().st_size,
                                mtime_ns=path.stat().st_mtime_ns) for sensor, year, path in sources])
atomic_json(parameters, parameters_path)
log = pd.DataFrame([dict(metric='HAR-12', reference_sensor='viirs-snpp', output_rows=len(result),
                         commit=COMMIT, parameters=json.dumps(parameters, sort_keys=True),
                         executed_at_utc=datetime.now(timezone.utc).isoformat())])
temporary = METRICS / '_log_har12_snpp.csv.tmp'
log.to_csv(temporary, index=False)
temporary.replace(METRICS / '_log_har12_snpp.csv')
print(result.to_string(index=False))
print('HAR-12 S-NPP outputs written to', METRICS)
